# LoG 2026 Rebuttal — Evaluation Suite (vast.ai)

Runs every experiment the rebuttal depends on. **Nothing here trains** — every
stage evaluates frozen checkpoints, so this is eval compute only.

Stages, in the order the rebuttal text blocks on:

| stage | what it answers |
|---|---|
| `smoke` | catches OOM / config breakage before the long jobs |
| `depth` | mistake-depth evidence: are errors single-node and terminal (algorithmic) or distributed mid-rollout (heuristic)? |
| `table1` | the statistics R1 asked for — Table 1 used 15 graphs/bin at 16-mixed |
| `wssweep` | does rewiring `p` alone explain the WS failure? includes shortcut error localisation |
| `disc` | soft vs hard re-injected mask hint, inference only |
| `path` | path graphs in fp32 to depth 1200 (~1153 sequential BFS steps) |

**Before running:** the checkpoints must be on this instance (see the checkpoint
cell), and the repo on this instance must contain `tests/run_rebuttal_suite.py`
— the verify cell checks both.

In [ ]:
import os
if os.path.exists('.git') and 'nar-experiments' in os.getcwd():
    !git pull
elif os.path.exists('nar-experiments'):
    %cd nar-experiments
    !git pull
else:
    !git clone https://github.com/MarkoMile/nar-experiments.git
    %cd nar-experiments
%pip install --no-cache-dir torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu129
%pip install --no-cache-dir torch-scatter -f https://data.pyg.org/whl/torch-2.8.0+cu129.html
%pip install --no-cache-dir pyg-lib -f https://data.pyg.org/whl/torch-2.8.0+cu129.html
%pip install --no-cache-dir tensorflow
%pip install --no-cache-dir torch-geometric
%pip install --no-cache-dir lightning
%pip install --no-cache-dir ogb yacs loguru wandb tabulate
%pip install --no-cache-dir "salsa-clrs @ git+https://github.com/jkminder/SALSA-CLRS.git"
%pip install --no-cache-dir "dm-clrs @ git+https://github.com/deepmind/clrs.git"
%pip install --no-cache-dir -q numpy scipy networkx ogb matplotlib tqdm scikit-learn

In [ ]:
import os
os.environ["PYTHONOPTIMIZE"] = "1"
os.environ["WANDB_CORE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

import torch, multiprocessing as mp
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
N_CORES = mp.cpu_count()
print("cores:", N_CORES)

## Verify the instance has what it needs

Graph generation is **serial by default** and dominates wall-clock at high sample
counts, so `--max-cores` is set to this box's core count below.

In [ ]:
import glob, subprocess, sys

# 1. the suite driver and the flags it depends on must be present
missing = [p for p in ["tests/run_rebuttal_suite.py", "tests/bfs_depth_analysis.py",
                       "tests/eval_checkpoint.py", "tests/eval_path.py"]
           if not os.path.exists(p)]
assert not missing, f"Missing on this instance: {missing}. Push them, or upload them here."

help_txt = subprocess.run([sys.executable, "tests/bfs_depth_analysis.py", "--help"],
                          capture_output=True, text=True).stdout
for flag in ("--precision", "--ws-p-sweep", "--shortcuts", "--mask-mode", "--max-cores"):
    assert flag in help_txt, f"{flag} missing — this instance has an old copy of the repo."
print("scripts OK")

# 2. checkpoints
CKPT_GLOB = "model-checkpoints/multiseed/*.ckpt"   # <-- edit to the grokked seeds
ckpts = sorted(glob.glob(CKPT_GLOB))
print(f"\n{len(ckpts)} checkpoint(s) matched {CKPT_GLOB}:")
for c in ckpts:
    print("  ", c, f"({os.path.getsize(c) / 1e6:.0f} MB)")
assert ckpts, "No checkpoints found. Upload them to model-checkpoints/ first."

## 1. Smoke test — do this before anything long

In [ ]:
!python -u tests/run_rebuttal_suite.py --ckpts "{CKPT_GLOB}" \
    --stages smoke --out-dir results/rebuttal --stop-on-fail

## 2. Full suite

Ordered so the results the rebuttal text needs land first. `path` runs last: it is
the longest (1153 sequential steps, batching does not help) and the least costly
to lose, since d16–d512 results already exist.

In [ ]:
# All checkpoints: the cheap, wide jobs. Dataset generation is cached across
# checkpoints (same seed + same generator kwargs), so it is paid once.
!python -u tests/run_rebuttal_suite.py --ckpts "{CKPT_GLOB}" \
    --stages depth table1 wssweep disc \
    --out-dir results/rebuttal \
    --num-samples 200 --sweep-samples 50 \
    --precisions 32 16-mixed \
    --sizes 800 1600 \
    --test-batch-size 5

## 2b. Path graphs — run LAST, on ONE checkpoint

1153 sequential rollout steps at d1200; batching does not help, so this is the
longest job and scales linearly in checkpoints. One checkpoint is enough for the
claim; d16-d512 results already exist from earlier runs.

In [ ]:
PATH_CKPT = "model-checkpoints/model-best.ckpt"   # the paper's checkpoint

!python -u tests/run_rebuttal_suite.py --ckpts "{PATH_CKPT}" \
    --stages path --out-dir results/rebuttal \
    --path-samples 15

## 3. W&B multiseed (per-seed table + grokking curves) — runs anywhere

In [ ]:
!python -u src/experiments/fetch_wandb_multiseed.py \
    --entity markomile-petnica \
    --project nar-experiments-finetuning \
    --group multiseed \
    --out-dir results/wandb/multiseed

## 4. Collect results, then stop the instance

In [ ]:
!python -u tests/summarize_rebuttal.py --results-dir results/rebuttal

In [ ]:
!tar czf rebuttal-results.tar.gz results/rebuttal results/wandb 2>/dev/null || true
print("Download rebuttal-results.tar.gz BEFORE stopping the instance.")
!ls -la rebuttal-results.tar.gz

In [ ]:
# !vastai set api-key <api_key>
# !vastai stop instance $CONTAINER_ID